# Exploratory analysis and sampling

**P0 Essential · D2 Independent · 80 minutes**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from pathlib import Path

def locate(relative: str, local_name: str | None = None) -> Path:
    candidates = []
    if local_name:
        candidates.append(Path.cwd() / local_name)
    candidates.extend(root / relative for root in [Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find {relative}. Run from the course clone or place the downloaded data beside this notebook."
    )


In [ ]:
path = locate('datasets/teaching/air-quality/observations.csv', 'observations.csv')
air = pd.read_csv(path)
air['timestamp'] = pd.to_datetime(air[['year','month','day','hour']])
air.shape

## Predict before plotting

Name one claim this three-station, fourteen-day sample can support and one it cannot. What dependence makes a random row split questionable for forecasting?

In [ ]:
def coverage_table(frame: pd.DataFrame) -> pd.DataFrame:
    return (frame.groupby('station', observed=True)
            .agg(rows=('timestamp','size'), start=('timestamp','min'), end=('timestamp','max'),
                 pm25_missing=('PM2.5', lambda s: int(s.isna().sum())))
            .reset_index())

In [ ]:
coverage = coverage_table(air)
assert set(coverage.columns) == {'station','rows','start','end','pm25_missing'}
assert coverage['rows'].sum() == len(air)
coverage

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for station, group in air.groupby('station'):
    daily = group.set_index('timestamp')['PM2.5'].resample('D').mean()
    ax.plot(daily.index, daily, marker='o', label=station)
ax.set(title='Daily mean PM2.5 in the teaching slice', ylabel='PM2.5 (µg/m³)', xlabel='Date')
ax.legend()
fig.autofmt_xdate()
plt.show()

## Independent brief

Create exactly three figures: coverage/missingness, a distribution with extremes, and one stratified relationship. Every caption must report the analysis count and one limitation. Do not claim that this slice represents all stations, seasons, cities, or current conditions.

**Instructor note.** Assess the claim/figure/sample connection, not visual decoration. Ask how missingness and autocorrelation change interpretation.